In [1]:
import os
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Morgan fingerprint generator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_id(smiles):
    if not isinstance(smiles, str) or pd.isna(smiles):
        return -1

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return -1

    fingerprint = morgan_gen.GetFingerprint(mol)
    return int(hashlib.sha256(fingerprint.ToBitString().encode("ascii")).hexdigest()[:16], 16) % (10**8)

# Resolve processed datasets from the repository root.
repo_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'datasets' / 'processed').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root containing datasets/processed.')
base_path = repo_root / 'datasets' / 'processed'
datasets = {
    'ALMANAC': 'almanac_final_without_last_score.xls',
    'DrugComb_LINCS_PRISM': 'DrugComb_LINCS_PRISM_final_without_last_score.zip',
    'DrugCombDB_LINCS_PRISM': 'DrugCombDBLINCS_PRISM_final_without_last_score.csv',
    'OncologyScreen_LINCS_PRISM': 'OncologyScreenLINCS_PRISM_final_without_last_score.xls',
    'ONEIL_LINCS_PRISM': 'OneilLINCS_PRISM_final_without_last_score.csv'
}

def load_dataset(file_path):
    if file_path.lower().endswith('.zip'):
        return pd.read_csv(file_path, compression='zip')

    if file_path.lower().endswith(('.xls', '.xlsx')):
        try:
            return pd.read_excel(file_path)
        except Exception:
            return pd.read_csv(file_path, sep=None, engine='python')

    return pd.read_csv(file_path)

def prepare_data(df):
    if 'synergy_score' not in df.columns:
        raise ValueError('synergy_score column was not found.')

    # Convert drug SMILES to Morgan fingerprint IDs
    for column in ['drugname1', 'drugname2']:
        if column in df.columns:
            df[column + '_id'] = df[column].apply(smiles_to_id)
            df.drop(columns=[column], inplace=True)

    # Convert remaining text columns, such as cellline, to numeric values
    for column in df.select_dtypes(include=['object', 'string']).columns:
        df[column] = df[column].astype('category').cat.codes

    y = pd.to_numeric(df['synergy_score'], errors='coerce').values
    X = df.drop(columns=['synergy_score']).apply(pd.to_numeric, errors='coerce').values

    valid_rows = np.isfinite(y)
    X = np.nan_to_num(X[valid_rows])
    y = y[valid_rows]
    return X, y

def train_and_evaluate(dataset_name, file_name):
    file_path = os.path.join(base_path, file_name)
    print('\n' + '=' * 60)
    print(f'PROCESSING: {dataset_name}')
    print('=' * 60)

    df = load_dataset(file_path)
    print(f'Dataset shape: {df.shape}')
    print('Generating Morgan fingerprint IDs...')
    X, y = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print('Training AdaBoost model...')
    base_tree = DecisionTreeRegressor(max_depth=10, random_state=42)
    model = AdaBoostRegressor(
        estimator=base_tree,
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f'MSE  : {mse:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'MAE  : {mae:.4f}')
    print(f'R2   : {r2:.4f}')

    return {
        'Dataset': dataset_name,
        'Samples': len(y),
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2 Score': r2
    }

results = []
for dataset_name, file_name in datasets.items():
    try:
        results.append(train_and_evaluate(dataset_name, file_name))
    except Exception as error:
        print(f'Could not train {dataset_name}: {error}')

results_df = pd.DataFrame(results)
print('\n' + '=' * 60)
print('FINAL ADABOOST RESULTS')
print('=' * 60)
print(results_df.to_string(index=False))
results_df



PROCESSING: ALMANAC
Dataset shape: (29251, 334)
Generating Morgan fingerprint IDs...


C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)
C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)


Training AdaBoost model...
MSE  : 1995.0098
RMSE : 44.6655
MAE  : 33.3535
R2   : 0.3604

PROCESSING: DrugComb_LINCS_PRISM
Dataset shape: (17139, 334)
Generating Morgan fingerprint IDs...


C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)
C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)


Training AdaBoost model...
MSE  : 20.6883
RMSE : 4.5484
MAE  : 3.3477
R2   : 0.2829

PROCESSING: DrugCombDB_LINCS_PRISM
Dataset shape: (17139, 334)
Generating Morgan fingerprint IDs...


C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)
C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)


... (2 outputs left out)

Training AdaBoost model...
MSE  : 363.8418
RMSE : 19.0746
MAE  : 13.6009
R2   : 0.3726

PROCESSING: ONEIL_LINCS_PRISM
Dataset shape: (2714, 334)
Generating Morgan fingerprint IDs...


C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)
C:\Users\Aashrith\AppData\Local\Temp\ipykernel_27372\2220797561.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column + '_id'] = df[column].apply(smiles_to_id)


Training AdaBoost model...
MSE  : 295.1161
RMSE : 17.1789
MAE  : 12.7034
R2   : 0.4107

FINAL ADABOOST RESULTS
                   Dataset  Samples         MSE      RMSE       MAE  R2 Score
                   ALMANAC    29251 1995.009810 44.665533 33.353493  0.360409
      DrugComb_LINCS_PRISM    17139   20.688260  4.548435  3.347703  0.282932
    DrugCombDB_LINCS_PRISM    17139   20.688260  4.548435  3.347703  0.282932
OncologyScreen_LINCS_PRISM     1160  363.841841 19.074639 13.600867  0.372575
         ONEIL_LINCS_PRISM     2714  295.116071 17.178943 12.703424  0.410741


,Dataset,Samples,MSE,RMSE,MAE,R2 Score
0,ALMANAC,29251,1995.009810,44.665533,33.353493,0.360409
1,DrugComb_LINCS_PRISM,17139,20.688260,4.548435,3.347703,0.282932
2,DrugCombDB_LINCS_PRISM,17139,20.688260,4.548435,3.347703,0.282932
3,OncologyScreen_LINCS_PRISM,1160,363.841841,19.074639,13.600867,0.372575
4,ONEIL_LINCS_PRISM,2714,295.116071,17.178943,12.703424,0.410741
